## Set Up

In [1]:
import pandas as pd
import json as js 
import numpy as np
import re
import requests 
from bs4 import BeautifulSoup
from time import sleep

## Load Datasets

In [2]:
curr_path = '/content/drive/MyDrive/Self-study/ML/melon/'
song_meta = pd.read_json(curr_path + 'melon_data/song_meta.json')
date_df = pd.read_csv(curr_path + 'melon_data/date.csv')

## Get `is_top` 

In [11]:
#get only song from 2017
issue_year_list = []
for date in song_meta['issue_date']:
    if date != 0:
        issue_year_list.append(int(str(date)[:4]))
    else:
        issue_year_list.append(np.nan)

song_meta['issue_year'] = issue_year_list
song_meta = song_meta[song_meta['issue_year'] >= 2017]


In [12]:
#remove song with various artists
idx = 0
idx_lst = []
for id in song_meta['artist_id_basket']:
    if len(id) < 2:
        idx_lst.append(idx)
    idx+=1

song_meta = song_meta.iloc[idx_lst,:]

In [13]:
#remove [] in `artist_id_basket` and `song_gn_gnr_basket`

artist_list = []
genre_list = []

for row_idx, row_series in song_meta.iterrows():
    try:
        artist_list.append(row_series[4][0])
    except:
        artist_list.append(np.nan)
    try:
        genre_list.append(row_series[6][0])
    except:
        genre_list.append(np.nan)

song_meta['artist_id'] = artist_list
song_meta['song_gn_gnr'] = genre_list

In [14]:
song_meta

,song_gn_dtl_gnr_basket,issue_date,album_name,album_id,artist_id_basket,song_name,song_gn_gnr_basket,artist_name_basket,id,issue_year,artist_id,song_gn_gnr
2,[GN0901],20180518,Hit,4698747,[3361],Solsbury Hill (Remastered 2002),[GN0900],[Peter Gabriel],2,2018.0,3361,GN0900
9,"[GN0105, GN0101]",20170320,Pastel Reflection,10047088,[753752],"사랑, 그대라는 멜로디",[GN0100],[진호],9,2017.0,753752,GN0100
10,[GN1201],20170407,Luv.Loops,10053652,[1625859],Hi (Heyoo),[GN1200],[Miraa.],10,2017.0,1625859,GN1200
16,[GN0901],20191023,Earth Glow,10341972,[896417],Can&#39;t Stand Still,[GN0900],[Ruelle],16,2019.0,896417,GN0900
18,"[GN2805, GN2801]",20190326,ASMR 백색소음 자장가 Best 8,10265451,[2636332],ASMR 숙면과 휴식에 좋은 편안한 빗소리 (백색소음),[GN2800],[ASMR 백색소음 (ASMR White Noise)],18,2019.0,2636332,GN2800
...,...,...,...,...,...,...,...,...,...,...,...,...
707957,"[GN0105, GN0101]",20171012,너를 그리워해,10101402,[1758500],너를 그리워해,[GN0100],[토요],707957,2017.0,1758500,GN0100
707970,"[GN0908, GN0901]",20181130,Peppermint,10227880,[406421],Peppermint,[GN0900],[Tiffany Young],707970,2018.0,406421,GN0900
707976,"[GN1301, GN1302]",20181018,Colors Compilation,2692383,[967093],Everybody (Just Bounce),[GN1300],[Vbnd],707976,2018.0,967093,GN1300
707978,"[GN0908, GN0901]",20191025,My Blood,10343254,[100377],My Blood,[GN0900],[Westlife],707978,2019.0,100377,GN0900


In [15]:
startDay_list = []

for row_idx, row_series in date_df.iterrows():
    startDay_list.append(row_series[5])

startDay_list = startDay_list[:-1]
file_path_list = [curr_path + 'melon_data/chart/{startDay}.csv'.format(startDay = day) for day in startDay_list]

In [16]:
all_charts_df = pd.DataFrame({'rank':[], 'song_name': [], 'song_id': [], 'artist_name':[], 'artist_id':[], 'alb_name' :[], 'alb_id':[]})

for file_path in file_path_list:
    additional_df = pd.read_csv(file_path)
    all_charts_df = all_charts_df.append(additional_df, ignore_index = True)

In [17]:
unique_songs_df = all_charts_df.drop_duplicates(['song_id'])

In [ ]:
song_meta

,song_gn_dtl_gnr_basket,issue_date,album_name,album_id,artist_id_basket,song_name,song_gn_gnr_basket,artist_name_basket,id,issue_year,artist_id,song_gn_gnr
2,[GN0901],20180518,Hit,4698747,[3361],Solsbury Hill (Remastered 2002),[GN0900],[Peter Gabriel],2,2018.0,3361,GN0900
9,"[GN0105, GN0101]",20170320,Pastel Reflection,10047088,[753752],"사랑, 그대라는 멜로디",[GN0100],[진호],9,2017.0,753752,GN0100
10,[GN1201],20170407,Luv.Loops,10053652,[1625859],Hi (Heyoo),[GN1200],[Miraa.],10,2017.0,1625859,GN1200
16,[GN0901],20191023,Earth Glow,10341972,[896417],Can&#39;t Stand Still,[GN0900],[Ruelle],16,2019.0,896417,GN0900
18,"[GN2805, GN2801]",20190326,ASMR 백색소음 자장가 Best 8,10265451,[2636332],ASMR 숙면과 휴식에 좋은 편안한 빗소리 (백색소음),[GN2800],[ASMR 백색소음 (ASMR White Noise)],18,2019.0,2636332,GN2800
...,...,...,...,...,...,...,...,...,...,...,...,...
707957,"[GN0105, GN0101]",20171012,너를 그리워해,10101402,[1758500],너를 그리워해,[GN0100],[토요],707957,2017.0,1758500,GN0100
707970,"[GN0908, GN0901]",20181130,Peppermint,10227880,[406421],Peppermint,[GN0900],[Tiffany Young],707970,2018.0,406421,GN0900
707976,"[GN1301, GN1302]",20181018,Colors Compilation,2692383,[967093],Everybody (Just Bounce),[GN1300],[Vbnd],707976,2018.0,967093,GN1300
707978,"[GN0908, GN0901]",20191025,My Blood,10343254,[100377],My Blood,[GN0900],[Westlife],707978,2019.0,100377,GN0900


In [ ]:
unique_songs_df

,rank,song_name,song_id,artist_name,artist_id,alb_name,alb_id,Unnamed: 0
0,1.0,당신의 밤 (Feat. 오혁),30179089.0,황광희 X 개코,1285544.0,무한도전 위대한 유산,10027428.0,0.0
1,2.0,에라 모르겠다,30147445.0,BIGBANG (빅뱅),198094.0,MADE,10022709.0,1.0
2,3.0,Beautiful,30157753.0,Crush,674710.0,도깨비 OST Part 4,10024106.0,2.0
3,4.0,좋다고 말해,30163110.0,볼빨간사춘기,792022.0,Full Album RED PLANET 'Hidden Track',10024816.0,3.0
4,5.0,Stay With Me,30132687.0,찬열 (CHANYEOL),672857.0,도깨비 OST Part.1,10020654.0,4.0
...,...,...,...,...,...,...,...,...
27992,96.0,인생찬가,35008534.0,임영웅,994944.0,IM HERO,10923444.0,95.0
27996,100.0,사랑역,35008529.0,임영웅,994944.0,IM HERO,10923444.0,99.0
28063,67.0,팡파레,35145136.0,다비치,236815.0,Season Note,10955743.0,66.0
28080,84.0,미친 것처럼,35126568.0,V.O.S,108794.0,아픔을 말하는,10954134.0,83.0


In [18]:
song_meta.rename(columns = {'album_id': 'alb_id'}, inplace = True)
main_df = song_meta.merge(unique_songs_df, how = 'outer', on = ['song_name', 'artist_id', 'alb_id'])

In [21]:
main_df

,song_gn_dtl_gnr_basket,issue_date,album_name,alb_id,artist_id_basket,song_name,song_gn_gnr_basket,artist_name_basket,id,issue_year,artist_id,song_gn_gnr,rank,song_id,artist_name,alb_name,Unnamed: 0
0,[GN0901],20180518.0,Hit,4698747.0,[3361],Solsbury Hill (Remastered 2002),[GN0900],[Peter Gabriel],2.0,2018.0,3361.0,GN0900,NaN,NaN,NaN,NaN,NaN
1,"[GN0105, GN0101]",20170320.0,Pastel Reflection,10047088.0,[753752],"사랑, 그대라는 멜로디",[GN0100],[진호],9.0,2017.0,753752.0,GN0100,NaN,NaN,NaN,NaN,NaN
2,[GN1201],20170407.0,Luv.Loops,10053652.0,[1625859],Hi (Heyoo),[GN1200],[Miraa.],10.0,2017.0,1625859.0,GN1200,NaN,NaN,NaN,NaN,NaN
3,[GN0901],20191023.0,Earth Glow,10341972.0,[896417],Can&#39;t Stand Still,[GN0900],[Ruelle],16.0,2019.0,896417.0,GN0900,NaN,NaN,NaN,NaN,NaN
4,"[GN2805, GN2801]",20190326.0,ASMR 백색소음 자장가 Best 8,10265451.0,[2636332],ASMR 숙면과 휴식에 좋은 편안한 빗소리 (백색소음),[GN2800],[ASMR 백색소음 (ASMR White Noise)],18.0,2019.0,2636332.0,GN2800,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171664,NaN,NaN,NaN,10923444.0,NaN,인생찬가,NaN,NaN,NaN,NaN,994944.0,NaN,96.0,35008534.0,임영웅,IM HERO,95.0
171665,NaN,NaN,NaN,10923444.0,NaN,사랑역,NaN,NaN,NaN,NaN,994944.0,NaN,100.0,35008529.0,임영웅,IM HERO,99.0
171666,NaN,NaN,NaN,10955743.0,NaN,팡파레,NaN,NaN,NaN,NaN,236815.0,NaN,67.0,35145136.0,다비치,Season Note,66.0
171667,NaN,NaN,NaN,10954134.0,NaN,미친 것처럼,NaN,NaN,NaN,NaN,108794.0,NaN,84.0,35126568.0,V.O.S,아픔을 말하는,83.0


In [32]:
main_df['is_top'] = np.where(main_df['rank'] != np.nan, 1, 0)

In [31]:
main_df = main_df.replace('nan', np.nan)

In [33]:
main_df

,song_gn_dtl_gnr_basket,issue_date,album_name,alb_id,artist_id_basket,song_name,song_gn_gnr_basket,artist_name_basket,id,issue_year,artist_id,song_gn_gnr,rank,song_id,artist_name,alb_name,Unnamed: 0,is_top
0,[GN0901],20180518.0,Hit,4698747.0,[3361],Solsbury Hill (Remastered 2002),[GN0900],[Peter Gabriel],2.0,2018.0,3361.0,GN0900,NaN,NaN,NaN,NaN,NaN,1
1,"[GN0105, GN0101]",20170320.0,Pastel Reflection,10047088.0,[753752],"사랑, 그대라는 멜로디",[GN0100],[진호],9.0,2017.0,753752.0,GN0100,NaN,NaN,NaN,NaN,NaN,1
2,[GN1201],20170407.0,Luv.Loops,10053652.0,[1625859],Hi (Heyoo),[GN1200],[Miraa.],10.0,2017.0,1625859.0,GN1200,NaN,NaN,NaN,NaN,NaN,1
3,[GN0901],20191023.0,Earth Glow,10341972.0,[896417],Can&#39;t Stand Still,[GN0900],[Ruelle],16.0,2019.0,896417.0,GN0900,NaN,NaN,NaN,NaN,NaN,1
4,"[GN2805, GN2801]",20190326.0,ASMR 백색소음 자장가 Best 8,10265451.0,[2636332],ASMR 숙면과 휴식에 좋은 편안한 빗소리 (백색소음),[GN2800],[ASMR 백색소음 (ASMR White Noise)],18.0,2019.0,2636332.0,GN2800,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171664,NaN,NaN,NaN,10923444.0,NaN,인생찬가,NaN,NaN,NaN,NaN,994944.0,NaN,96.0,35008534.0,임영웅,IM HERO,95.0,1
171665,NaN,NaN,NaN,10923444.0,NaN,사랑역,NaN,NaN,NaN,NaN,994944.0,NaN,100.0,35008529.0,임영웅,IM HERO,99.0,1
171666,NaN,NaN,NaN,10955743.0,NaN,팡파레,NaN,NaN,NaN,NaN,236815.0,NaN,67.0,35145136.0,다비치,Season Note,66.0,1
171667,NaN,NaN,NaN,10954134.0,NaN,미친 것처럼,NaN,NaN,NaN,NaN,108794.0,NaN,84.0,35126568.0,V.O.S,아픔을 말하는,83.0,1
